# Semester Assignment 1

Group A Members:
- Philo Odermatt 521059
- Jonah Sontag 410828
- Siqi He 519748
- Pascal Kising 375086

In [1]:
import numpy as np
import pandas as pd

# Load and display the data for missing values
df = pd.read_excel('../Aufgabe_1_Gloss_Optimization.xlsx')
print(f"Shape: {df.shape},\nMissing Values: \n{df.isna().sum()}")

Shape: (58, 15),
Missing Values: 
V13       0
V15       0
V16       0
V18       9
V19       0
V21      43
V28       0
V29       0
V30       0
V31       0
V32       0
V33       0
V34       0
V39       0
gloss     0
dtype: int64


## 1. Data Preparation

V18 and V21 are only dosed if used. Wherever the type is missing, the corresponding dosage (V16 respectively V19) is exactly zero, which confirms that a missing value means the additive was not used. The missing values are therefore filled with the explicit category 'none' instead of being dropped.

Categorical columns are one hot encoded and numerical columns are standardized. Since gloss values are in the thousands, the target is also standardized via TransformedTargetRegressor so that scale sensitive models (LASSO, SVR, MLP) work on reasonable scale. All preprocessing is placed inside the pipeline so that it is refitted on each cross validation fold and no information leaks from validation folds into training.

In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# missing addivite type means the additive was not used
for c in ["V18", "V21"]:
    df[c] = df[c].fillna("none")

X = df.drop(columns="gloss")
y = df["gloss"]

cat_cols = ["V15", "V18", "V21", "V39"]
num_cols = [c for c in X.columns if c not in cat_cols]

# random 10% hold out test set, data is not time series, so random split is fine
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, shuffle=True)

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def make_model(reg):
    # scales features and target, keeps everything inside the CV loop
    return TransformedTargetRegressor(
        Pipeline([("prep", prep), ("reg", reg)]),
        transformer=StandardScaler())

## 2. Model Selection and Hyperparameter Tuning

In [3]:
# three models spanning increasing complexity: linear, ensemble, neural network
models = {
    "LASSO": (make_model(Lasso(max_iter=100000)),
              {"regressor__reg__alpha": np.logspace(-3, 1, 30)}),
    "Random Forest": (make_model(RandomForestRegressor(random_state=42)),
                      {"regressor__reg__n_estimators": [100, 300],
                       "regressor__reg__max_depth": [3, 5, 10, None],
                       "regressor__reg__min_samples_leaf": [1, 2, 4]}),
    "MLP": (make_model(MLPRegressor(solver="lbfgs", max_iter=20000,
                                    random_state=42)),
            {"regressor__reg__hidden_layer_sizes": [(16,), (32,), (64,), (32, 16)],
             "regressor__reg__activation": ["relu", "tanh"],
             "regressor__reg__alpha": [1e-4, 1e-2, 0.1, 1]})}

rows = []
for name, (model, grid) in models.items():
    gs = GridSearchCV(model, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    rows.append({
        "model": name,
        "CV RMSE": -gs.best_score_,
        "CV RMSE Std": gs.cv_results_["std_test_score"][gs.best_index_],
        "Test RMSE": mean_squared_error(y_test, y_pred) ** 0.5,
        "Test MAE": mean_absolute_error(y_test, y_pred),
        "Test R2": r2_score(y_test, y_pred)
    })
    print(name, gs.best_params_)

results = pd.DataFrame(rows).set_index("model").round(3)
print("\nResults:\n", results)

LASSO {'regressor__reg__alpha': np.float64(0.003562247890262444)}
Random Forest {'regressor__reg__max_depth': None, 'regressor__reg__min_samples_leaf': 1, 'regressor__reg__n_estimators': 300}
MLP {'regressor__reg__activation': 'tanh', 'regressor__reg__alpha': 0.1, 'regressor__reg__hidden_layer_sizes': (32,)}

Results:
                 CV RMSE  CV RMSE Std  Test RMSE  Test MAE  Test R2
model                                                             
LASSO          1219.823      349.530    950.912   859.384    0.663
Random Forest  1467.167      233.935   1065.833   904.222    0.577
MLP            1222.727      318.770   1081.967  1001.667    0.564


## 3. Discussion

Three regression models were chosen to cover increasing levels of model complexity: LASSO as a reguralized linear model, Random Forest as a nonlinear ensemble of decision trees and a multilayer perceptron as a neural network. LASSO was preferred over OLS as the linear model representative because the one hot encoded design matrix has roughly 39 columns for only 52 training samples, where unreguralized OLS is rather unstable, while the L1 penalty removes uninformative columns.

Model selection is based primarily on the cross validation RMSE, since there are only 6 observations in the test set and a single score on 6 points is not statistically reliable on its own.

LASSO achieved the best CV RMSE (1219.8 ± 349.5) and the best test performance (RMSE 950.9, R² 0.66). Random Forest achieved a clearly worse CV RMSE (1467.2 ± 233.9) and a slightly worse test performance (RMSE 1065.8, R² 0.58), so its additional flexibility does not pay off on this dataset. The MLP produced a CV RMSE comparable to LASSO (1222.7 ± 318.8), but performed slightly worse on the test set (RMSE 1082.0, R² 0.56).

The comparison across the models shows that increasing model complexity does not improve performance here. Following the requirement that the model should only be as complex as necessary, LASSO is selected as the final model. It has the best cross validated error and is the simplest of the three models. Additionally, its sparse coefficients remain interpretable, which is valuable for gloss optimization since it directly shows which variables drive gloss. With gloss values ranging from 2600 to 13600 a typical prediction error of about 1000 gloss units means the model captures the main trends but is not precise enough for very fine optimization. The main limitation is the sample size, thus all metrics carry considerable variance and gathering more experimental data would be the most effective way to improve the model. 